# Version 3 — Semantic Memory: Pinecone Knowledge Base + PostgreSQL

The agent searches an existing Pinecone knowledge base before it writes SQL, so it uses real table and column names instead of guessing.

## Workflow

```text
User Question
     |
     v
AI Data Analyst Agent
     |
     +---- lookup_schema(term) ---------> Pinecone: nlq-schema    (tables, columns, joins)
     |
     +---- lookup_examples(question) ---> Pinecone: nlq-examples  (similar question + SQL)
     |
     +---- lookup_common(question) -----> Pinecone: common        (business definitions)
     |
     v
Generate read-only SQL
     |
     v
run_query(sql) -------------------------> PostgreSQL
     |                                      |
     +<------------- query results ---------+
     |
     v
Final Answer grounded in query results
```

**Tools:** three Pinecone retrieval tools + `run_query(sql)`.

The agent picks the lookups it needs. It does not call all three every time.

## What changes from Version 2

| | Version 2 | Version 3 |
|---|---|---|
| Knows the schema | Table names in the prompt | Looks it up in `nlq-schema` |
| Similar solved questions | — | `lookup_examples` → `nlq-examples` |
| Business definitions | — | `lookup_common` → `common` |
| Table list in the prompt | Yes | **Removed** — the agent looks it up |
| Chat model | OpenAI `gpt-6-astra` | **AWS Bedrock** (`BEDROCK_CHAT_MODEL_ID`) |
| OpenAI is used for | the chat model | **query embeddings only** |

**Why the model change:** the knowledge base and the rest of the project run on AWS, so the chat model moves to Bedrock. OpenAI stays only because the Pinecone vectors were made with an OpenAI embedding model, and query vectors must come from the same model.

**Why semantic search:** a user says “buyers”; the schema says `customers`. A user says “money kept”; the definition uses payment status. Embedding search matches meaning, not exact words.

## 1. Install dependencies

In [ ]:
%pip install -q langchain langchain-aws langchain-openai pinecone python-dotenv "psycopg[binary]"

## 2. Configure the model, knowledge base and database

This notebook **reads** the existing Pinecone index `ecom-kb`. It does not build or upload anything.

| Variable | Required | Used for |
|---|---|---|
| `PINECONE_API_KEY` | yes | Reads the `ecom-kb` index |
| `PINECONE_INDEX_NAME` | no (default `ecom-kb`) | Index name |
| `PINECONE_EMBEDDING_MODEL` | no (default `text-embedding-3-small`) | Must match the model used to fill the index |
| `OPENAI_API_KEY` | yes | Query embeddings only |
| `BEDROCK_CHAT_MODEL_ID` | yes | The chat model (the brain) |
| `AWS_REGION` | no (default `us-east-1`) | Bedrock region; AWS credentials come from your normal AWS config |
| `PGHOST`, `PGDATABASE`, `PGUSER`, `PGPASSWORD` | yes | PostgreSQL connection (use a read-only role) |
| `PGPORT` | no (default `5432`) | |
| `PGSSLMODE` | no | e.g. `require` if your provider needs TLS |

Put these in a `.env` file next to the notebook. Never commit credentials.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuration — Version 3
#
#   .env ──► load_dotenv ──► checks ──► model        (Bedrock chat model: the brain)
#                                    ├─► embeddings   (OpenAI: turns a search phrase into a vector)
#                                    ├─► index        (Pinecone `ecom-kb`: the knowledge base)
#                                    └─► PostgreSQL settings (used by the SQL tools)
#
# The embedding model MUST be the same one used when the vectors were
# written to Pinecone, otherwise similarity scores are meaningless.
#
# Does NOT build or upload the knowledge base. It reads the existing index.
# ─────────────────────────────────────────────────────────────
import os
import json
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "ecom-kb")
EMBEDDING_MODEL = os.getenv("PINECONE_EMBEDDING_MODEL", "text-embedding-3-small")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_CHAT_MODEL_ID = os.getenv("BEDROCK_CHAT_MODEL_ID", "")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY.")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY for query embeddings. It must match the embedding model used to build the Pinecone index.")
if not BEDROCK_CHAT_MODEL_ID:
    raise ValueError("Set BEDROCK_CHAT_MODEL_ID to a chat model enabled in your AWS account.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.getenv(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.getenv("PGPORT", "5432"))
PGSSLMODE = os.getenv("PGSSLMODE")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pinecone.Index(PINECONE_INDEX_NAME)
model = ChatBedrockConverse(
    model=BEDROCK_CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

## 3. Carried over: PostgreSQL helpers and `run_query`

These cells are unchanged from Version 2. Run them; the explanation is in Version 2.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PostgreSQL helpers (shared by the SQL tools)
#
#   pg_connect()   ──► new connection, read-only by default, with a statement timeout
#   format_rows()  ──► columns + rows ──► plain-text table the model can read
#
# Does NOT replace a read-only database role. The role is the real
# security boundary; these settings are a second layer.
# ─────────────────────────────────────────────────────────────
MAX_ROWS = 50


def pg_connect(timeout_ms: int = 15000):
    """Open a PostgreSQL connection whose transactions are read-only."""
    kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
        "options": f"-c default_transaction_read_only=on -c statement_timeout={timeout_ms}",
    }
    if PGSSLMODE:
        kwargs["sslmode"] = PGSSLMODE
    return psycopg.connect(**kwargs)


def format_rows(columns: list[str], rows: list[dict], truncated: bool) -> str:
    """Render rows as 'col | col' lines. States clearly when rows were cut off."""
    if not rows:
        return "Query succeeded. 0 rows returned."
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(row.get(c)) for c in columns) for row in rows)
    tail = (f"({len(rows)} rows shown. More rows exist; the result was capped at {MAX_ROWS}.)"
            if truncated else f"({len(rows)} rows.)")
    return f"{header}\n{body}\n{tail}"

In [ ]:
# ─────────────────────────────────────────────────────────────
# run_query(sql) — the agent's HANDS
#
#   sql ──► STEP 1 first-word gate (SELECT / WITH only)
#       ──► STEP 2 execute on a read-only connection
#       ──► STEP 3 format up to 50 rows as text ──► back to the model
#
# Errors are returned as text ("SQL error: ...") so the model can read
# them and fix its query. The tool never raises into the agent loop.
#
# Does NOT check for multiple statements or hidden writes. That is a
# known gap in this version; Version 4 adds a proper SQL policy.
# ─────────────────────────────────────────────────────────────
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SELECT or WITH query against PostgreSQL.

    Returns at most 50 rows as text, or a message starting with
    "SQL error:" or "REFUSED:" that explains what went wrong.
    """
    # STEP 1 — first-word gate.
    sql_clean = sql.strip().rstrip(";").strip()
    if not sql_clean:
        return "REFUSED: SQL query is empty."
    first_word = sql_clean.split(None, 1)[0].upper()
    if first_word not in {"SELECT", "WITH"}:
        return "REFUSED: Only read-only SELECT or WITH queries are allowed."

    # STEP 2 — execute. Fetch one extra row so we know if the result was cut off.
    try:
        with pg_connect() as conn:
            cursor = conn.execute(sql_clean)
            fetched = cursor.fetchmany(MAX_ROWS + 1)
            columns = [d.name for d in cursor.description] if cursor.description else []
    except psycopg.Error as exc:
        return f"SQL error: {exc}"

    # STEP 3 — format for the model.
    return format_rows(columns, fetched[:MAX_ROWS], truncated=len(fetched) > MAX_ROWS)

## 4. New: the knowledge-base tools — the agent's memory

| Tool | Namespace | Answers |
|---|---|---|
| `lookup_schema(term)` | `nlq-schema` | Which tables and columns mean what? |
| `lookup_examples(question)` | `nlq-examples` | Has a similar question been solved, and with what SQL? |
| `lookup_common(question)` | `common` | What is the business definition (e.g. revenue)? |

Each tool embeds the search text, asks Pinecone for the closest records, and returns their metadata as text. Vectors are never sent to the model.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Knowledge-base tools — the agent's MEMORY
#
#   lookup_schema(term)        ──► search_namespace("nlq-schema")   ──► tables, columns, joins
#   lookup_examples(question)  ──► search_namespace("nlq-examples") ──► similar question + SQL
#   lookup_common(question)    ──► search_namespace("common")       ──► business definitions
#
#   search_namespace: text ──► embedding ──► Pinecone top-k ──► [{id, score, metadata}]
#   format_matches:   matches ──► readable text (no vectors), cut to max_chars
#
# Does NOT execute SQL and does NOT write to Pinecone.
# ─────────────────────────────────────────────────────────────


def search_namespace(namespace: str, text: str, top_k: int = 8) -> list[dict]:
    """Embed `text` and return the top-k matches from one Pinecone namespace."""
    vector = embeddings.embed_query(text)
    response = index.query(vector=vector, namespace=namespace, top_k=top_k, include_metadata=True)

    # The Pinecone client can return a dict or an object; read both shapes.
    matches = response.get("matches", []) if isinstance(response, dict) else response.matches
    results = []
    for match in matches:
        if isinstance(match, dict):
            results.append({"id": match.get("id", "unknown"),
                            "score": match.get("score"),
                            "metadata": match.get("metadata") or {}})
        else:
            results.append({"id": getattr(match, "id", "unknown"),
                            "score": getattr(match, "score", None),
                            "metadata": getattr(match, "metadata", None) or {}})
    return results


def format_matches(matches: list[dict], max_chars: int = 12000) -> str:
    """Turn matches into text blocks: 'RESULT n | id=... | score=...' plus metadata lines."""
    if not matches:
        return "No matching knowledge-base records were found."
    blocks = []
    for rank, match in enumerate(matches, start=1):
        lines = [f"RESULT {rank} | id={match['id']} | score={match['score']}"]
        for key, value in match["metadata"].items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            lines.append(f"{key}: {value}")
        blocks.append("\n".join(lines))
    output = "\n\n---\n\n".join(blocks)
    return output[:max_chars] + ("\n...[truncated]" if len(output) > max_chars else "")


@tool
def lookup_schema(term: str) -> str:
    """Search the nlq-schema Pinecone namespace for tables and columns.
    Use it to confirm real table/column names, data types, descriptions, synonyms,
    and business meaning. Retrieves metadata only; does not execute SQL."""
    return format_matches(search_namespace("nlq-schema", term, top_k=8))


@tool
def lookup_examples(question: str) -> str:
    """Search the nlq-examples Pinecone namespace for similar approved NL-to-SQL examples.
    Use examples as patterns, then verify schema details using lookup_schema.
    Retrieves examples only; does not execute SQL."""
    return format_matches(search_namespace("nlq-examples", question, top_k=4), max_chars=10000)


@tool
def lookup_common(question: str) -> str:
    """Search the common Pinecone namespace for shared business definitions or guidance.
    Retrieves knowledge only; does not execute SQL."""
    return format_matches(search_namespace("common", question, top_k=4), max_chars=8000)

## 5. Test the lookups on their own

Call the tools directly first. This shows what the knowledge base returns, before the agent is involved.

In [ ]:
# Call each retrieval tool directly (no agent, no SQL) to see what the knowledge base returns.
print(lookup_schema.invoke({"term": "product category identifier"}))
print("\n--- Similar SQL examples ---\n")
print(lookup_examples.invoke({"question": "top products by units sold"}))
print("\n--- Common knowledge ---\n")
print(lookup_common.invoke({"question": "how should revenue be calculated?"}))

## 6. The system prompt

The table list is gone. The prompt describes the three namespaces and says **when** to use each lookup.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
The existing Pinecone index `ecom-kb` contains three knowledge namespaces:
- `nlq-schema`: schema metadata, columns, relationships, and related descriptions.
- `nlq-examples`: similar natural-language questions and SQL examples.
- `common`: shared business definitions and guidance.
PostgreSQL is the source of truth for internal marketplace data.

AVAILABLE TOOLS
- lookup_schema(term): searches Pinecone namespace `nlq-schema`.
- lookup_examples(question): searches Pinecone namespace `nlq-examples`.
- lookup_common(question): searches Pinecone namespace `common`.
- run_query(sql): executes a read-only SQL query against PostgreSQL and returns up
  to 50 rows.
This version does not have a dedicated SQL validation tool.

TOOL-SELECTION AND WORKFLOW
- Use lookup_schema when table/column names, joins, grain, or definitions are not
  already established by reliable context.
- Use lookup_examples when a similar approved question or SQL pattern may help.
- Use lookup_common when shared business definitions or guidance may affect the answer.
- Do not call every retrieval tool automatically; choose those relevant to the question.
- Treat retrieved examples as patterns, not guaranteed-correct answers. Verify columns,
  joins, filters, date ranges, and aggregation against retrieved schema.
- Generate one read-only SELECT/WITH query for internal data questions, then call
  run_query(sql). Inspect the result before answering.
- If SQL fails, use the error and retrieved knowledge to correct it when possible.
- Do not claim a query succeeded unless run_query confirms it.
- If schema details are insufficient, say what is missing rather than inventing them.

CONSTRAINTS
- Never invent numbers, results, schema details, or business definitions.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state the numerator and denominator.
- If evidence is insufficient, state UNKNOWN and explain why.
- Do not expose private chain-of-thought; provide a concise explanation only.
- Read-only restrictions must also be enforced by database permissions.

OUTPUT FORMAT
Use relevant sections:
ANSWER: Answer with units where applicable, or UNKNOWN.
DENOMINATOR: For percentages, identify numerator and denominator; otherwise N/A.
SOURCE: Identify PostgreSQL and include the SQL actually executed, if successful.
CAVEAT: State material limitations, ambiguity, or data-quality concerns.
"""

## 7. Create the agent with four tools

In [ ]:
agent = create_agent(
    model=model,
    tools=[lookup_schema, lookup_examples, lookup_common, run_query],
    system_prompt=SYSTEM_PROMPT,
)

## 8. Run the agent and watch the loop

Look for this order in the trace: a lookup first, then `run_query`, then the answer.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ask_agent(question) — run the agent once and print what happened
#
#   question ──► agent.invoke ──► final state
#                                   │
#                                   ├─ AIMessage with tool_calls ──► "BRAIN → TOOL"
#                                   ├─ ToolMessage               ──► "TOOL → OBSERVATION"
#                                   └─ last message              ──► "FINAL ANSWER"
#
# Does NOT change the agent. It only reads the messages it returns.
# ─────────────────────────────────────────────────────────────
def ask_agent(question: str, agent_to_use=None, max_obs_chars: int = 1500):
    agent_to_use = agent_to_use or agent
    result = agent_to_use.invoke({"messages": [{"role": "user", "content": question}]})

    line = "=" * 80
    print(f"{line}\nQUESTION\n{line}\n{question}")
    print(f"\n{line}\nAGENT TRACE\n{line}")

    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "AIMessage" and getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("\n🧠 BRAIN → TOOL")
                print("Tool:", call["name"])
                print("Arguments:", call["args"])
        elif kind == "ToolMessage":
            print(f"\n🖐️ TOOL → OBSERVATION ({message.name})")
            print(str(message.content)[:max_obs_chars])

    print(f"\n{line}\nFINAL ANSWER\n{line}")
    print(result["messages"][-1].content)
    if "sql_call_count" in result:
        print("\nrun_query executions used:", result["sql_call_count"])
    return result

In [ ]:
result = ask_agent("What are the top 5 products by units sold?")

In [ ]:
result = ask_agent("What percentage of orders were cancelled?")

## 9. Key points

| Tool | Purpose | Role |
|---|---|---|
| `lookup_*` | Retrieves knowledge about the data | **Memory** |
| `run_query(sql)` | Retrieves the data itself | **Hands** |
| Chat model | Chooses tools and reads results | **Brain** |

- Lookups tell the agent **how the data is structured**. `run_query` tells it **what the data contains**.
- Retrieved examples are patterns, not answers. The agent must still check them against the schema.
- `top_k` (8 / 4 / 4) are starting values. Test them with questions whose correct tables you already know.

**Still missing:** nothing checks the SQL before it runs. A bad query goes straight to the database.

**Next — Version 4:** `validate_query(sql)` checks every query before `run_query` runs it.